# Phase 1: Data acquisition, exploration and cleaning
TinyStoriesInstruct sample: 15,000 stories, 5,000 each of Dialogue, MoralValue and Conflict.

Run `python data/download_tinystories.py` and `python src/clean_pipeline.py` from the repo root before running this notebook.

In [ ]:
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import spacy
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.clean_pipeline import (FEATURES_RE, HEADER_RE, DIALOGUE_RE,
                                clean_text, extract_label)

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 120)

## 1. Corpus statistics

In [ ]:
df = pd.read_csv(ROOT / "data" / "raw_tinystories.csv")
print(df.shape)
df["n_words"] = df["story"].str.split().str.len()
print(df["n_words"].describe().round(1))
print(df.groupby("narrative_type")["n_words"].mean().round(1))
print(df["narrative_type"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["n_words"], bins=60, color="steelblue")
ax.set_xlabel("Story length (words)")
ax.set_ylabel("Number of stories")
ax.set_title("Story length distribution")
fig.savefig(FIG_DIR / "length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Vocabulary and Zipf's law

In [ ]:
WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")
all_tokens = []
for story in df["story"]:
    text = story.lower().replace("’", "'")
    all_tokens.extend(WORD_RE.findall(text))
counts = Counter(all_tokens)
n_tokens = len(all_tokens)
n_types = len(counts)
hapax = sum(1 for c in counts.values() if c == 1)
print(f"Tokens: {n_tokens:,}")
print(f"Types (vocabulary): {n_types:,}")
print(f"Type/token ratio: {n_types / n_tokens:.4f}")
print(f"Hapax legomena: {hapax:,} ({hapax / n_types:.1%} of types)")
print(counts.most_common(20))

In [ ]:
freqs = np.array(sorted(counts.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
for n in [100, 1000, len(freqs)]:
    slope = np.polyfit(np.log10(ranks[:n]), np.log10(freqs[:n]), 1)[0]
    print(f"Zipf slope over top {n:>5} ranks: {slope:.2f}")
print(f"Top 10 words cover {freqs[:10].sum() / n_tokens:.1%} of all tokens")

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ranks, freqs, label="TinyStories")
ax.loglog(ranks, freqs[0] / ranks, "--", label="ideal Zipf (1/rank)")
ax.set_xlabel("Rank (log)")
ax.set_ylabel("Frequency (log)")
ax.legend()
fig.savefig(FIG_DIR / "zipf.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Messy text examples

In [ ]:
print(df.loc[0, "raw_text"])

In [ ]:
MESSY_PATTERNS = {
    "curly quotes": r"[“”’]",
    "long dash": r"[—–]",
    "ellipsis": r"…|\.{3,}",
    "double space": r"  ",
    "summary after story": r"\nSummary:[^\n]*\n<\|endoftext",
}
for name, pattern in MESSY_PATTERNS.items():
    hits = df["raw_text"].str.contains(pattern, regex=True)
    first = df.index[hits][0]
    m = re.search(pattern, df.loc[first, "raw_text"])
    snippet = df.loc[first, "raw_text"][max(0, m.start() - 60): m.end() + 60]
    print(f"--- {name}: {hits.sum()} stories (example id {first})")
    print(repr(snippet))

## 4. Tokenizer comparison (regex vs NLTK vs spaCy)

In [ ]:
nltk.download("punkt_tab", quiet=True)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
REGEX_TOKEN_RE = re.compile(r"\w+(?:'\w+)?|[^\w\s]")

samples = [
    '"I don\'t want to go," said Lily.',
    "Tim’s mom made ice-cream... It was yummy!",
    "Ben—the big dog—ran fast.",
]
for s in samples:
    print("TEXT :", s)
    print("REGEX:", REGEX_TOKEN_RE.findall(s))
    print("NLTK :", word_tokenize(s))
    print("SPACY:", [tok.text for tok in nlp(s)])
    print()

## 5. The four regex patterns

In [ ]:
example = df.loc[0, "raw_text"]
print("Regex 1 label:", FEATURES_RE.search(example).group(1))
print("Regex 2 header removed:\n", HEADER_RE.sub("", example)[:200])
print("Label check:", (df["raw_text"].apply(extract_label) == df["narrative_type"]).mean())

df["story_clean"] = df["raw_text"].apply(clean_text)
df["n_quotes"] = df["story_clean"].apply(lambda t: len(DIALOGUE_RE.findall(t)))
print(df.groupby("narrative_type")["n_quotes"].mean().round(2))
print(DIALOGUE_RE.findall(df.loc[df["n_quotes"] > 0, "story_clean"].iloc[0])[:3])

## 6. Stop-word decision

In [ ]:
stops = nlp.Defaults.stop_words
print(f"spaCy has {len(stops)} stop words")
for w in ["i", "you", "we", "not", "no"]:
    print(f"{w!r} in stop list: {w in stops}")
for w in ["i", "you", "we", "not"]:
    rate = df.groupby("narrative_type")["story_clean"].apply(
        lambda s: s.str.lower().str.count(rf"\b{w}\b").mean())
    print(w, rate.round(2).to_dict())

## 7. Stemming vs lemmatization

In [ ]:
stemmer = PorterStemmer()
words = ["cried", "jumped", "puppies", "happily", "running", "was", "flies", "studies"]
print("Stem :", [stemmer.stem(w) for w in words])
print("Lemma:", [nlp(w)[0].lemma_ for w in words])
doc = nlp("Lily cried because the puppies were running and the flies were buzzing.")
print([(t.text, t.lemma_) for t in doc if t.is_alpha])

## 8. Vocabulary size before and after cleaning
Needs `data/cleaned_tinystories.csv` from `python src/clean_pipeline.py`.

In [ ]:
clean = pd.read_csv(ROOT / "data" / "cleaned_tinystories.csv")
vocab_raw = Counter(tok for text in df["raw_text"] for tok in text.split())
vocab_mid = Counter(tok for text in df["story_clean"] for tok in WORD_RE.findall(text.lower()))
vocab_final = Counter(tok for text in clean["text_processed"].fillna("") for tok in text.split())
summary = pd.DataFrame({
    "stage": ["raw (whitespace split)", "after noise removal + lowercase", "after lemmatization"],
    "vocabulary size": [len(vocab_raw), len(vocab_mid), len(vocab_final)],
})
print(summary.to_string(index=False))
clean.head()